# IF3270 Machine Learning | Tubes 2

## Import Libraries

In [1]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, TensorDataset

from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, Dense, LSTM, RNN, SimpleRNN, Dropout, TimeDistributed
from tensorflow.keras.callbacks import EarlyStopping

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Reproducibility
SEED = 42
def set_reproducibility(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    tf.keras.utils.set_random_seed(seed)
    tf.config.experimental.enable_op_determinism()

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_reproducibility(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')


Using device: cpu


## Import Dataset

In [ ]:
# Download dataset
import kagglehub

output_dir = './dataset'

if not os.path.exists(output_dir):
    kagglehub.login()

    intel_path = kagglehub.dataset_download(
        handle="puneet6060/intel-image-classification",
        force_download=True,
        output_dir=os.path.join(output_dir, "intel-image-classification")
        )
    print("Path to dataset intel-image-classification files:", intel_path)

    flickr8k_path = kagglehub.dataset_download(
        handle="adityajn105/flickr8k",
        force_download=True,
        output_dir=os.path.join(output_dir, "flickr8k")
    )
    print("Path to dataset flickr8k files:", flickr8k_path)

100%|██████████| 346M/346M [00:41<00:00, 8.79MB/s] 

Extracting files...


Path to dataset intel-image-classification files: ./dataset\intel-image-classification


100%|██████████| 1.04G/1.04G [02:05<00:00, 8.86MB/s]

Extracting files...


Path to dataset flickr8k files: ./dataset\flickr8k


In [5]:
DATA_DIR = './dataset/intel-image-classification/seg_train/seg_train'
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2, # 20% untuk validasi
    subset="training",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)

# Pembuatan Validation Set
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)

Found 14034 files belonging to 6 classes.
Using 11228 files for training.
Found 14034 files belonging to 6 classes.
Using 2806 files for validation.


---
# 1. Exploratory Data Analysis (EDA)

---
# 2. Data Cleaning and Preprocessing

---
# 3. Modeling and Validation


In [4]:
cnn_encoder = Sequential([
    Input(shape=(224, 224, 3)),
    tf.keras.layers.Rescaling(1./255),
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.5)
])

cnn_encoder.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 394272)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │   100,933,888 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 100,934,784 (385.04 MB)

 Trainable params: 100,934,784 (385.04 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
cnn_encoder.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
k_history = cnn_encoder.fit(x_train_k, y_train_k, epochs=10, validation_data=(x_val_k, y_val_k), batch_size=64)

---
# 4. Error Analysis

---
# 5. Insights